### Exercise 6: Intent-Based Routing (Middleware)


Topic: Middleware & RunnableBranch


Goal: Swap system prompts based on user input.

In [1]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [4]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableBranch, RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain.chat_models import init_chat_model

llm = init_chat_model("groq:openai/gpt-oss-120b")
# Create a classification chain that classifies the user query into "tech" or "billing".
classification_prompt = ChatPromptTemplate.from_messages([
    ("system", "Classify the user query into one word: tech or billing."),
    ("human", "{input}"),
])

classification_chain = classification_prompt | llm | StrOutputParser()

# technical support chain
tech_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a Senior Developer. Use technical terms."),
    ("human", "{input}")
])

tech_chain = tech_prompt | llm | StrOutputParser()

# billing support chain
billing_prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a polite Billing Agent. Help with invoices."), 
    ("human", "{input}")
])

billing_chain = billing_prompt | llm | StrOutputParser()

# Create a router that routes to the appropriate chain based on the classification.
router = RunnableBranch(
    (lambda x: 'tech' in x['topic'].lower(), tech_chain),
    (lambda x: 'billing' in x['topic'].lower(), billing_chain),
    billing_chain
)

# Create a full chain that first classifies the query and then routes to the appropriate chain.
full_chain = (
    RunnableLambda(lambda x: {"input": x, "topic": classification_chain.invoke({'input': x})}) | router
)

# Test the full chain with different queries.
print('Tech question:')
print(full_chain.invoke("How do I reset my api key"))

# Test the full chain with a billing question.
print('\nBilling question:')
print(full_chain.invoke("Where is my receipt?"))

Tech question:
Below is a **step‑by‑step guide** (and a few code snippets) for safely resetting an API key.  
I’ll cover the **generic workflow** that applies to most SaaS platforms (OpenAI, AWS, GCP, Azure, etc.) and then give a concrete **example for the OpenAI platform**, which you can adapt to your own service.

---

## 1. High‑level rotation workflow

| Phase | Action | Why it matters |
|-------|--------|----------------|
| **A. Preparation** | • Identify every place the key is used (CI/CD pipelines, local dev, production services, secret stores).<br>• Ensure you have a *new* key ready before you revoke the old one. | Prevents downtime and broken integrations. |
| **B. Generate a new key** | • Use the provider’s UI, CLI, or API to create a fresh secret. | Fresh secret = fresh entropy, revokes any leaked material. |
| **C. Propagate the new key** | • Update environment variables, config files, secret‑manager entries, Docker secrets, etc.<br>• Deploy the change to a **staging** envi